In [1]:
'''
tutorial:
https://www.kaggle.com/code/alexandervc/gene-ontology-python-tutorial
'''

'\ntutorial:\nhttps://www.kaggle.com/code/alexandervc/gene-ontology-python-tutorial\n'

In [2]:
from goatools import obo_parser

In [3]:
# obo file contains the go term database
go_obo = 'data/go-basic.obo'

# reads obo file to dictionary
go = obo_parser.GODag(go_obo)

data/go-basic.obo: fmt(1.2) rel(2026-07-26) 41,378 Terms


In [4]:
go_id = 'GO:0048527'
go_term = go[go_id]

In [5]:
go_term

GOTerm('GO:0048527'):
  id:GO:0048527
  item_id:GO:0048527
  name:lateral root development
  namespace:biological_process
  _parents: 1 items
    GO:0048528
  parents: 1 items
    GO:0048528	level-04	depth-06	post-embryonic root development [biological_process]
  children: 0 items
  level:5
  depth:7
  is_obsolete:False
  alt_ids: 0 items

In [ ]:
'GO term name: {}'.format(go_term.name)

In [ ]:
rec = go[go_id]
rec

In [ ]:
# get parents of a go term
parents = rec.get_all_parents()
parents

In [ ]:
# get all childrn of a go term
children = rec.get_all_children()
children

In [ ]:
for term in parents.union(children):
    print(go[term])

In [ ]:
# section 4 go enrichment or depletion analysis
from goatools.go_enrichment import GOEnrichmentStudy

In [ ]:
# Sanbomics tutorial
'''
https://www.youtube.com/watch?v=ONiWugVEf2s&t=1s
'''

In [2]:
#this part is not important, only a simple example of scanpy processing to get a list of genes
import numpy as np
import scanpy as sc
import pandas as pd
from matplotlib.pyplot import rc_context

my_data_path = 'path/to/data/outs/filtered_feature_bc_matrix'

def pp(path):
    adata = sc.read_10x_mtx(path)
    sc.pp.filter_cells(adata, min_genes=200) #get rid of cells with fewer than 200 genes
    sc.pp.filter_genes(adata, min_cells=3) #get rid of genes that are found in fewer than 3 cells
    adata.var['mt'] = adata.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, .98)
    lower_lim = np.quantile(adata.obs.n_genes_by_counts.values, .02)
    adata = adata[(adata.obs.n_genes_by_counts < upper_lim) & (adata.obs.n_genes_by_counts > lower_lim)]
    adata = adata[adata.obs.pct_counts_mt < 20]
    sc.pp.normalize_total(adata, target_sum=1e4) #normalize every cell to 10,000 UMI
    sc.pp.log1p(adata) #change to log counts
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5) #these are default values
    adata.raw = adata #save raw data before processing values and further filtering
    adata = adata[:, adata.var.highly_variable] #filter highly variable
    sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt']) #Regress out effects of total counts per cell and the percentage of mitochondrial genes expressed
    sc.pp.scale(adata, max_value=10) #scale each gene to unit variance
    sc.tl.pca(adata, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20)
    sc.tl.leiden(adata, resolution = 0.25)
    sc.tl.umap(adata)
    sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
    
    
    #find markers
    results = adata.uns['rank_genes_groups']
    out = np.array([[0,0,0,0,0]])
    for group in results['names'].dtype.names:
        out = np.vstack((out, np.vstack((results['names'][group],
                                         results['scores'][group],
                                         results['pvals_adj'][group],
                                         results['logfoldchanges'][group],
                                         np.array([group] * len(results['names'][group])).astype('object'))).T))
    markers = pd.DataFrame(out[1:], columns = ['Gene', 'scores', 'pval_adj', 'lfc', 'cluster'])
    adata.uns['markers'] = markers #save marker df to uns

    
    return adata

In [3]:
# c elegans ncbi taxonomy id is 6239
# paste this into search at https://www.ncbi.nlm.nih.gov/gene 
# "6239"[Taxonomy ID] AND alive[property] AND genetype protein coding[Properties]
# send to > file > text file > create file
# get gene_result.txt
# this is the background gene set

In [4]:
# find this in environment from goatools
# /home/carl/miniconda3/envs/ontology/bin/ncbi_gene_results_to_python.py

In [5]:
# -o output
!python /home/carl/miniconda3/envs/ontology/bin/ncbi_gene_results_to_python.py -o genes_ncbi_c_elegans_proteincoding.py gene_result.txt

      19,983 lines READ:  gene_result.txt
      19,983 geneids WROTE: genes_ncbi_c_elegans_proteincoding.py


In [24]:
# import from genes_ncbi_c_elegans_proteincoding.py
# can move file to python path
from genes_ncbi_c_elegans_proteincoding import GENEID2NT as GeneID2nt_elegans

In [25]:
from goatools.base import download_go_basic_obo
from goatools.base import download_ncbi_associations
from goatools.obo_parser import GODag
from goatools.anno.genetogo_reader import Gene2GoReader
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS

In [26]:
obo_fname = download_go_basic_obo()
fin_gene2go = download_ncbi_associations()
obodag = GODag("go-basic.obo")

  EXISTS: go-basic.obo
  EXISTS: gene2go
go-basic.obo: fmt(1.2) rel(2026-07-26) 41,378 Terms


In [27]:
# test find daf-16 gene id: 172981
GeneID2nt_elegans[172981]

ntncbi(tax_id=6239, Org_name='Caenorhabditis elegans', GeneID=172981, CurrentID=0, Status='live', Symbol='daf-16', Aliases=['CELE_R13H8.1'], description='Forkhead box protein O', other_designations='Forkhead box protein O', map_location='', chromosome='I', genomic_nucleotide_accession_version='NC_003279.8', start_position_on_the_genomic_accession=10750498, end_position_on_the_genomic_accession=10776703, orientation='plus', exon_count=15, OMIM=[], no_hdr0='')

In [28]:
# new dictionary
# gene names and their id
mapper = {}

for key in GeneID2nt_elegans:
    mapper[GeneID2nt_elegans[key].Symbol] = GeneID2nt_elegans[key].GeneID

# mapper but swap keys and values
inv_map = {v: k for k, v in mapper.items()}

In [29]:
mapper

{'homt-1': 171590,
 'nlp-40': 171591,
 'rcor-1': 171592,
 'sesn-1': 171593,
 'pgs-1': 171594,
 'Y48G1C.5': 171595,
 'Y48G1C.6': 171597,
 'pid-2': 171599,
 'rab-11.1': 171601,
 'rpl-7': 171602,
 'F53G12.9': 171603,
 'F53G12.8': 171604,
 'col-45': 171605,
 'spe-8': 171606,
 'mex-3': 171607,
 'bli-3': 171608,
 'ptr-11': 171609,
 'cest-27': 171610,
 'F56C11.3': 171611,
 'snpc-3.2': 171615,
 'marc-4': 171616,
 'ztf-3': 171617,
 'C53D5.1': 171618,
 'C53D5.5': 171619,
 'xpo-2': 171621,
 'nol-14': 171622,
 'daf-25': 171623,
 'mbtr-1': 171624,
 'Y48G1A.2': 171625,
 'rnp-8': 171627,
 'R119.3': 171628,
 'taf-4': 171630,
 'R119.5': 171631,
 'otub-2': 171633,
 'C07F11.2': 171634,
 'tol-1': 171635,
 'W04C9.4': 171636,
 'cutl-13': 171637,
 'W04C9.2': 171638,
 'haf-4': 171639,
 'Y65B4BL.3': 171640,
 'Y65B4BL.4': 171641,
 'deps-1': 171642,
 'acs-13': 171643,
 'Y65B4BL.1': 171644,
 'psf-3': 171645,
 'icd-2': 171646,
 'wwp-1': 171647,
 'lpr-1': 171648,
 'grl-16': 171649,
 'hum-7': 171650,
 'eme-1': 17165

In [32]:
fin_gene2go

'gene2go'

In [31]:
objanno = Gene2GoReader(fin_gene2go, taxids=[6239])

HMS:0:00:00.000184       0 annotations,      0 genes,      0 GOs, 0 taxids READ: gene2go 


AssertionError: **FATAL: NO TAXIDS: gene2go